In [ ]:
#the import packages
import requests
import pandas as pd
from pandas import json_normalize
import requests
import os
from pathlib import Path
from datetime import datetime, timezone,timedelta,time
from scipy import stats
import json
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
import sklearn.metrics as metrics

In [ ]:
pd.set_option("display.max_columns", None)

In [ ]:
def loadDataFromFile(file_name):
    script_dir = Path().resolve().parent

    data_folder = script_dir / 'dataAnalysis and machine learning'/'data'
    print(data_folder)
    data_folder.mkdir(exist_ok=True)
    
    file_path = data_folder / (file_name + ".json")
    
    if file_path.exists():
        df = pd.read_json(file_path)
        print(f"Loaded {len(df)} records from {file_path}")
        return df
    else:
        print(f"File {file_path} does not exist.")
        return None    

In [ ]:
userInputData = loadDataFromFile("UserPrevious experiments-preprocessed")
timeSeriesData_BIG = loadDataFromFile("Data:Previous experiments-preprocessed")

In [ ]:
userInputData

In [ ]:
timeSeriesData_BIG

In [ ]:
# Convert back to timedelta
timeSeriesData_BIG['timestamp'] = pd.to_timedelta(timeSeriesData_BIG['timestamp'], unit='ms')

# Convert back to datetime

timeSeriesData_BIG ["datetime_timestamp"]= timeSeriesData_BIG['datetime_timestamp'].transform(
    lambda x: pd.to_datetime(x, unit='ms')
)
# Split back into dict
dict_of_timeseries = {k: v.drop(columns="keys").reset_index(drop=True) 
             for k, v in timeSeriesData_BIG.groupby("keys")}

In [ ]:
dict_of_timeseries[0]

In [ ]:
columns_datetime= [
       'date of experiment', 'actual timestamp StartingExperiment',
       'actual timestamp EndingExperiment', ]
columns_timedelta = ['time taken total',
       'timestamp InsertingSource timedelta',
       'time taken after insertion']

userInputData.loc[:,columns_datetime] = userInputData.loc[:,columns_datetime].apply(lambda x:pd.to_datetime(x, unit='ms'))
userInputData.loc[:,columns_timedelta] = userInputData.loc[:,columns_timedelta].apply(lambda x:pd.to_timedelta(x, unit='ms'))

In [ ]:
#remove all 
timeSeriesData_BIG_check_Interpolation = timeSeriesData_BIG.copy()
print(timeSeriesData_BIG_check_Interpolation.columns)

timeSeriesData_BIG_check_Interpolation["VOC given real values"] = timeSeriesData_BIG_check_Interpolation["VOC"]
timeSeriesData_BIG_check_Interpolation["VOC given real values"] = timeSeriesData_BIG_check_Interpolation["VOC given real values"].mask(timeSeriesData_BIG_check_Interpolation["original_value"]==False)
#timeSeriesData_BIG_check_Interpolation["points to check interpolation"] = timeSeriesData_BIG_check_Interpolation["VOC"]
#timeSeriesData_BIG_check_Interpolation["points to check interpolation"] = timeSeriesData_BIG_check_Interpolation["points to check interpolation"].mask(timeSeriesData_BIG_check_Interpolation["kept to test interpolation"] == False)
timeSeriesData_BIG_check_Interpolation["original_value_cum_sum"] = timeSeriesData_BIG_check_Interpolation.groupby(["keys","sensors"])["original_value"].transform("cumsum")
#make every second appear of a true given value null also 
#mask = (timeSeriesData_BIG_check_Interpolation["original_value"] == True) & (timeSeriesData_BIG_check_Interpolation["original_value_cum_sum"] % 2 == 0)
#timeSeriesData_BIG_check_Interpolation.loc[:,"kept to test interpolation"] =timeSeriesData_BIG_check_Interpolation.apply(lambda x: True if ((x["original_value"] == True) & (x["original_value_cum_sum"] % 2 == 1)) else False,axis=1)
timeSeriesData_BIG_check_Interpolation.head(20)

In [ ]:
# Split back into dict
dict_of_timeseries_inter = {k: v.drop(columns="keys").reset_index(drop=True) 
             for k, v in timeSeriesData_BIG_check_Interpolation.groupby("keys")}

In [ ]:
dict_of_timeseries_inter[65]

In [ ]:
plt.figure(figsize=(14, 6))

sns.lineplot(data = dict_of_timeseries_inter[71],x="seconds" ,y="VOC" ,hue="sensors")

In [ ]:
plt.figure(figsize=(14, 6))

sns.lineplot(data = dict_of_timeseries_inter[71],x="seconds" ,y="VOC given real values" ,hue="sensors")

In [ ]:
def insertColumnsTointerpolation_differences(interpolation_differences,method):
    columns= []
    sensors_names = ["Id=0:BME680:breathVocEquivalent","Id=1:BME680:breathVocEquivalent","Id=2:BME680:breathVocEquivalent"]
    columns.extend(["MSE-" + x +"-"+method for x in sensors_names])
    columns.extend(["MSE-" + "3 points missing-" + x +"-"+method for x in sensors_names])
    columns.extend(["MAE-" + x +"-"+method for x in sensors_names])
    columns.extend(["MSE-" + "3 points missing-" + x +"-"+method for x in sensors_names])
  #  columns.extend(["R2-" + x +"-"+method for x in sensors_names])
    for column in columns:
        interpolation_differences[column] = np.nan
    return interpolation_differences

In [ ]:
def checkInterpolationMethod(dict_of_timeseries_inter,interpolation_differences,method):
    insertColumnsTointerpolation_differences(interpolation_differences,method)
    #print(interpolation_differences)

    for index, row in userInputData.iterrows():
        
        experiment_df = dict_of_timeseries_inter[index]
        #print(experiment_df)
        sensors = experiment_df["sensors"].unique()
        target_column = "VOC-" +method
        target_column_3_points_missing = target_column + "-3 points missing"
        experiment_df[target_column] = experiment_df.apply(lambda x:x["VOC given real values"] if (x["original_value_cum_sum"]% 2 == 0) else np.nan,axis=1)
        three_points_missings = experiment_df.loc[experiment_df["VOC given real values"].notna()].copy()        
        three_points_missings[target_column_3_points_missing]  =  three_points_missings.apply(lambda x:x["VOC given real values"] if (x["original_value_cum_sum"]% 3 == 0) else np.nan,axis=1)
        for sensor in sensors:
            mask = (experiment_df["sensors"] == sensor)
          #  print(experiment_df.loc[mask,:].shape)
            experiment_df.loc[mask,target_column] = experiment_df.loc[mask,target_column].interpolate(method = method)
            three_points_missings.loc[mask,target_column_3_points_missing] = three_points_missings.loc[mask,target_column_3_points_missing].interpolate(method = method)
            dropped_df = experiment_df.loc[mask,:].dropna()
          #  print(dropped_df.shape)
            dropped_df_three_points_missings = experiment_df.loc[mask,:].dropna()

            mae = metrics.mean_absolute_error(dropped_df.loc[mask,"VOC given real values"], dropped_df.loc[mask,target_column])
            column_to_save = "MAE-"+sensor +"-"+method
            interpolation_differences.at[index,column_to_save] = mae
            mse = metrics.mean_squared_error(dropped_df.loc[mask,"VOC given real values"], dropped_df.loc[mask,target_column])
            column_to_save = "MSE-"+sensor +"-"+method
            interpolation_differences.at[index,column_to_save] = mse

            #############################
            three_points_missings.loc[mask,target_column_3_points_missing] = three_points_missings.loc[mask,target_column_3_points_missing].interpolate(method = method)
            dropped_df_three_points_missings = three_points_missings.loc[mask,:].dropna()

            
            mae = metrics.mean_absolute_error(dropped_df_three_points_missings.loc[mask,"VOC given real values"].dropna(), dropped_df_three_points_missings.loc[mask,target_column_3_points_missing].dropna())
            column_to_save = "MAE-"+ "3 points missing-" +sensor +"-"+method
            interpolation_differences.at[index,column_to_save] = mae
            mse = metrics.mean_squared_error(dropped_df_three_points_missings.loc[mask,"VOC given real values"].dropna(), dropped_df_three_points_missings.loc[mask,target_column_3_points_missing].dropna())
            column_to_save = "MSE-"+"3 points missing-" +sensor +"-"+method
            interpolation_differences.at[index,column_to_save] = mse
            
        dict_of_timeseries_inter[index] = experiment_df
    return dict_of_timeseries_inter   

In [ ]:
# Split back into dict
dict_of_timeseries_inter = {k: v.drop(columns="keys").reset_index(drop=True) 
             for k, v in timeSeriesData_BIG_check_Interpolation.groupby("keys")}
interpolation_differences = pd.DataFrame(index = np.arange(userInputData.shape[0]))
dict_of_timeseries_inter = checkInterpolationMethod(dict_of_timeseries_inter,interpolation_differences,"linear")
dict_of_timeseries_inter = checkInterpolationMethod(dict_of_timeseries_inter,interpolation_differences,"cubicspline")

In [ ]:
interpolation_differences

In [ ]:
#remove outliers
Q3_5_interpolation_differences = interpolation_differences.quantile(0.95)

interpolation_differences.where(interpolation_differences<Q3_5_interpolation_differences).mean()

In [ ]:
dict_of_timeseries_inter[71]

In [ ]:
plt.figure(figsize=(18, 6))

sns.lineplot(data = dict_of_timeseries_inter[71],x="seconds" ,y="VOC" ,hue="sensors")

In [ ]:
plt.figure(figsize=(18, 6))

sns.lineplot(data = dict_of_timeseries_inter[71],x="seconds" ,y="VOC-linear" ,hue="sensors")

In [ ]:
plt.figure(figsize=(18, 6))

sns.lineplot(data = dict_of_timeseries_inter[71],x="seconds" ,y="VOC-cubicspline" ,hue="sensors")